# מפגש 12 — SVM על Penguins: מבחירת מאפיינים ועד הערכת מודל
מחברת מודרכת ליחידה 3 המבוססת על SVM_Penguin_classification_exercise_3. בניגוד למפגש הקודם נבנה הפעם תהליך מסווג מלא.

**שאלת חקר:** האם מדידות גוף של פינגווינים מאפשרות לחזות את המין הביולוגי (`sex`)? זהו תרגיל חיזוי על נתוני מחקר, לא קביעה ביולוגית כללית.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

df=pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
display(df.head())
print(df.isna().sum())

## 1 — X ו־y
במחברת המקור היעד הוא `sex`, ולא `species`. נשתמש בארבע מדידות מספריות בלבד. נסיר שורות שבהן התגית חסרה, אך נלמד השלמת מדידות חסרות מתוך Train בלבד.

In [ ]:
features=["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
df=df.dropna(subset=["sex"]).copy()
X=df[features]
y=df["sex"]
print(y.value_counts())

## 2 — חלוקה בטוחה
Test יישמר עד להערכת המודל הסופית. `stratify=y` שומר בקירוב על יחס הקטגוריות.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
print(X_train.shape,X_test.shape)

## 3 — Pipeline ו־GridSearchCV
השלמת חסרים ותקנון מתבצעים בתוך כל קיפול אימון; GridSearchCV בוחר `C` ואת סוג ה־kernel בעזרת Train בלבד.

In [ ]:
pipeline=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("svc",SVC())])
params={"svc__C":[0.1,1,10],"svc__kernel":["linear","rbf"]}
search=GridSearchCV(pipeline,params,cv=5,scoring="accuracy")
search.fit(X_train,y_train)
print("Best CV parameters:",search.best_params_)
print("Best CV accuracy:",round(search.best_score_,3))

## 4 — הערכה סופית על Test
רק לאחר בחירת הפרמטרים נחשב מדדים על Test. Accuracy לבדו אינו מספיק אם יש חוסר איזון או עלויות שונות לטעויות.

In [ ]:
pred=search.predict(X_test)
print("Test accuracy:",round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()

## 5 — מסקנה וכרטיס יציאה
1. למה לא בחרנו פרמטרים על Test?
2. למה Pipeline עוזר למנוע דליפת מידע?
3. למה `sex` אינה ב־X?
4. האם שיפור Accuracy מוכיח קשר סיבתי?
5. הציעו מגבלה של המאגר והסבירו איך הייתם מתעדים את ההחלטות בפרויקט.